# 戦略5: 増配候補銘柄の抽出モデル

**作成日**: 2026-02-21 09:30

**目的**: 増配候補を予測するモデルの有効性を検証する

**検証内容**:
- 5つのポートフォリオ（ベースライン、低PBR、高ROE、過去リターン、複合）のパフォーマンス比較
- 増配予測精度の評価

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

print("ライブラリインポート完了")

## 1. データ読み込み・前処理

In [ ]:
PROJECT_ROOT = Path(r'C:\Users\yongr\claude project\workspace')

# 価格データ読み込み
print("価格データ読み込み中...")
df_price = pd.read_parquet(PROJECT_ROOT / 'data/curated/jquants/prices/daily_quotes_all.parquet')
df_price['date'] = pd.to_datetime(df_price['date'])
df_price = df_price[df_price['date'] >= '2017-01-01'].copy()
print(f"価格データ: {len(df_price):,} 行")

# 財務データ読み込み
print("財務データ読み込み中...")
df_fin = pd.read_parquet(PROJECT_ROOT / 'data/curated/jquants/financials/statements_all.parquet')
df_fin['disclosed_date'] = pd.to_datetime(df_fin['disclosed_date'])
df_fin = df_fin[df_fin['disclosed_date'] >= '2016-01-01'].copy()

# 年次決算のみに限定
if 'fiscal_quarter' in df_fin.columns:
    before_count = len(df_fin)
    df_fin = df_fin[df_fin['fiscal_quarter'] == 'FY'].copy()
    print(f"年次決算フィルタ: {before_count:,} → {len(df_fin):,} 行")
else:
    print("警告: fiscal_quarter列が存在しません")

print(f"財務データ: {len(df_fin):,} 行")
print("前処理完了")

In [ ]:
# 財務データのカラム確認
print("財務データのカラム:")
print(df_fin.columns.tolist())
print(f"\n行数: {len(df_fin):,}")
print(f"\nサンプルデータ:")
display(df_fin.head())

## 2. 必要な指標の計算

In [ ]:
# 必要なカラムを抽出（配当データと予想データを含む）
required_cols = ['disclosed_date', 'code', 'equity', 'net_profit', 'bps', 'DivAnn']

# FNP（当期純利益予想）があるかチェック
if 'FNP' in df_fin.columns:
    required_cols.append('FNP')
    has_fnp = True
else:
    print("警告: FNP（当期純利益予想）カラムが存在しません")
    has_fnp = False

# 存在するカラムのみ抽出
available_cols = [col for col in required_cols if col in df_fin.columns]
df_fin_clean = df_fin[available_cols].copy()

print(f"使用するカラム: {available_cols}")

# ROE計算
df_fin_clean['roe'] = (df_fin_clean['net_profit'] / df_fin_clean['equity']) * 100

# 予想ROE計算（FNPがある場合）
if has_fnp:
    df_fin_clean['forecast_roe'] = (df_fin_clean['FNP'] / df_fin_clean['equity']) * 100
else:
    # FNPがない場合、実績ROEを代用
    df_fin_clean['forecast_roe'] = df_fin_clean['roe']
    print("FNPがないため、forecast_roeは実績ROEで代用します")

# 異常値除外
df_fin_clean = df_fin_clean[
    (df_fin_clean['roe'] > -100) & (df_fin_clean['roe'] < 100) &
    (df_fin_clean['forecast_roe'] > -100) & (df_fin_clean['forecast_roe'] < 100) &
    (df_fin_clean['bps'] > 0) & (df_fin_clean['equity'] > 0)
].copy()

print(f"\n異常値除外後: {len(df_fin_clean):,} 行")
print("指標計算完了")

## 3. 価格データのピボット化

In [ ]:
print("価格データをピボット化中...")
df_price_pivot = df_price.pivot(index='date', columns='code', values='adjusted_close')
print(f"ピボットテーブル: {df_price_pivot.shape[0]} 日 × {df_price_pivot.shape[1]} 銘柄")
print("ピボット化完了")

## 4. リバランス日生成（月次）

In [ ]:
# 月次リバランス日（月末営業日）
trading_days = pd.DataFrame({'date': df_price_pivot.index})
trading_days['year'] = trading_days['date'].dt.year
trading_days['month'] = trading_days['date'].dt.month
rebalance_dates = trading_days.groupby(['year', 'month'])['date'].max().values
rebalance_dates = pd.Series(rebalance_dates).sort_values().reset_index(drop=True)

print(f"リバランス日数: {len(rebalance_dates)}")
print(f"期間: {rebalance_dates.iloc[0].date()} ~ {rebalance_dates.iloc[-1].date()}")

## 5. 財務データの事前処理

In [ ]:
print("財務データの事前処理中（時間がかかります）...")

# 各リバランス日での利用可能な財務データを事前計算
fin_by_date = {}

for i, rdate in enumerate(rebalance_dates):
    if i % 20 == 0:
        print(f"  進捗: {i}/{len(rebalance_dates)}")
    
    # その日までに開示された財務データ
    available = df_fin_clean[df_fin_clean['disclosed_date'] <= rdate].copy()
    
    # 各銘柄の最新データ
    latest = available.sort_values('disclosed_date').groupby('code').tail(1)
    
    # 配当データがある銘柄のみ
    if 'DivAnn' in latest.columns:
        latest = latest[latest['DivAnn'].notna()].copy()
    
    latest = latest.set_index('code')[['bps', 'roe', 'forecast_roe', 'DivAnn'] if 'DivAnn' in latest.columns else ['bps', 'roe', 'forecast_roe']]
    
    fin_by_date[rdate] = latest

print("財務データ事前処理完了")

## 6. 過去6ヵ月リターンの計算

In [ ]:
print("過去6ヵ月リターンを計算中...")

# 6ヵ月前の株価を取得するための関数
def get_return_6m(date, prices_pivot, lookback_days=130):
    """
    6ヵ月前（約130営業日前）の株価からのリターンを計算
    """
    if date not in prices_pivot.index:
        return pd.Series(dtype=float)
    
    # 現在の株価
    current_prices = prices_pivot.loc[date]
    
    # 6ヵ月前の日付を取得
    date_idx = prices_pivot.index.get_loc(date)
    lookback_idx = max(0, date_idx - lookback_days)
    past_date = prices_pivot.index[lookback_idx]
    past_prices = prices_pivot.loc[past_date]
    
    # リターン計算
    returns = (current_prices - past_prices) / past_prices
    
    return returns

# 各リバランス日の6ヵ月リターンを事前計算
returns_6m_by_date = {}

for i, rdate in enumerate(rebalance_dates):
    if i % 20 == 0:
        print(f"  進捗: {i}/{len(rebalance_dates)}")
    
    returns_6m = get_return_6m(rdate, df_price_pivot)
    returns_6m_by_date[rdate] = returns_6m

print("過去6ヵ月リターン計算完了")

## 7. 銘柄選定関数

In [ ]:
def screen_stocks(rebalance_date, prices_pivot, fin_data, returns_6m_data, strategy, n_stocks=20):
    """
    銘柄選定関数
    
    strategy:
        'baseline': ランダム（低PBR銘柄から）
        'low_pbr': B/P上位（低PBR）
        'high_roe': 予想ROE上位
        'positive_return': 過去6ヵ月リターン > 0
        'composite': 低PBR × 高ROE × 過去リターン > 0
    """
    # その日の価格
    if rebalance_date not in prices_pivot.index:
        return pd.DataFrame()
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    
    # 財務データ
    if rebalance_date not in fin_data:
        return pd.DataFrame()
    
    fin = fin_data[rebalance_date]
    
    # 6ヵ月リターン
    if rebalance_date not in returns_6m_data:
        return pd.DataFrame()
    
    returns_6m = returns_6m_data[rebalance_date]
    
    # マージ
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'bps': fin['bps'],
        'roe': fin['roe'],
        'forecast_roe': fin['forecast_roe'],
        'return_6m': returns_6m
    }).dropna()
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # PBRとB/P計算
    merged['pbr'] = merged['adjusted_close'] / merged['bps']
    merged['bp'] = merged['bps'] / merged['adjusted_close']  # B/P = 1/PBR
    
    # 異常値除外
    merged = merged[(merged['pbr'] > 0.01) & (merged['pbr'] < 50)]
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # 戦略別の銘柄選定
    if strategy == 'baseline':
        # ランダム選定（再現性のためseedを設定）
        selected = merged.sample(n=min(n_stocks, len(merged)), random_state=42)
    
    elif strategy == 'low_pbr':
        # B/P上位（低PBR）
        selected = merged.nlargest(n_stocks, 'bp')
    
    elif strategy == 'high_roe':
        # 予想ROE上位
        selected = merged.nlargest(n_stocks, 'forecast_roe')
    
    elif strategy == 'positive_return':
        # 過去6ヵ月リターン > 0の銘柄から20銘柄
        candidates = merged[merged['return_6m'] > 0]
        if len(candidates) < n_stocks:
            # 候補が少ない場合はリターン上位を選定
            selected = merged.nlargest(n_stocks, 'return_6m')
        else:
            # リターン上位から選定
            selected = candidates.nlargest(n_stocks, 'return_6m')
    
    elif strategy == 'composite':
        # 低PBR × 高ROE × 過去リターン > 0
        pbr_q1 = merged['pbr'].quantile(0.25)
        roe_q3 = merged['forecast_roe'].quantile(0.75)
        
        candidates = merged[
            (merged['pbr'] <= pbr_q1) &
            (merged['forecast_roe'] >= roe_q3) &
            (merged['return_6m'] > 0)
        ]
        
        if len(candidates) < n_stocks:
            # 候補が少ない場合は条件緩和（低PBR × 高ROE）
            candidates = merged[
                (merged['pbr'] <= pbr_q1) &
                (merged['forecast_roe'] >= roe_q3)
            ]
        
        if len(candidates) < n_stocks:
            # さらに少ない場合は低PBRのみ
            candidates = merged.nsmallest(n_stocks, 'pbr')
        else:
            # B/P上位から選定
            candidates = candidates.nlargest(n_stocks, 'bp')
        
        selected = candidates.head(n_stocks)
    
    else:
        return pd.DataFrame()
    
    return selected.reset_index()

# テスト
test_date = rebalance_dates.iloc[10]
test_result = screen_stocks(test_date, df_price_pivot, fin_by_date, returns_6m_by_date, 'composite')
print(f"テスト（複合戦略）: {len(test_result)} 銘柄選定")
display(test_result.head())

## 8. バックテスト関数

In [ ]:
def backtest_strategy(strategy_name, rebalance_dates, prices_pivot, fin_data, returns_6m_data, 
                     initial_cash=10_000_000, n_stocks=20, tax_rate=0.20315, unit=100):
    """
    バックテスト実行関数
    """
    cash = initial_cash
    portfolio = {}
    annual_realized_pnl = 0
    current_year = None
    results = []
    
    print(f"\n{'='*60}")
    print(f"戦略: {strategy_name}")
    print(f"{'='*60}")
    
    for i, rebalance_date in enumerate(rebalance_dates):
        if i % 20 == 0:
            print(f"進捗: {i}/{len(rebalance_dates)} ({i/len(rebalance_dates)*100:.1f}%) - {rebalance_date.date()}")
        
        # 年の切り替わり
        if current_year != rebalance_date.year:
            if current_year is not None and annual_realized_pnl > 0:
                tax = annual_realized_pnl * tax_rate
                cash -= tax
            annual_realized_pnl = 0
            current_year = rebalance_date.year
        
        # 既存ポートフォリオ売却
        sell_value = 0
        if rebalance_date in prices_pivot.index:
            for code, position in portfolio.items():
                if code in prices_pivot.columns:
                    sell_price = prices_pivot.loc[rebalance_date, code]
                    if pd.notna(sell_price):
                        sell_amount = position['shares'] * sell_price
                        sell_value += sell_amount
                        pnl = (sell_price - position['buy_price']) * position['shares']
                        annual_realized_pnl += pnl
        
        cash += sell_value
        portfolio = {}
        
        # 銘柄選定
        selected = screen_stocks(rebalance_date, prices_pivot, fin_data, returns_6m_data, strategy_name, n_stocks)
        
        if len(selected) == 0:
            results.append({
                'date': rebalance_date,
                'cash': cash,
                'n_stocks': 0,
                'invested': 0,
                'annual_pnl': annual_realized_pnl,
                'portfolio': {}
            })
            continue
        
        # 購入
        target_per_stock = cash / len(selected)
        total_invested = 0
        
        for _, row in selected.iterrows():
            code = row['code']
            price = row['adjusted_close']
            shares = int(target_per_stock / (price * unit)) * unit
            
            if shares > 0:
                invest_amount = shares * price
                total_invested += invest_amount
                portfolio[code] = {'shares': shares, 'buy_price': price}
        
        cash -= total_invested
        
        results.append({
            'date': rebalance_date,
            'cash': cash,
            'n_stocks': len(portfolio),
            'invested': total_invested,
            'annual_pnl': annual_realized_pnl,
            'portfolio': portfolio.copy()
        })
    
    # 最終税金
    if annual_realized_pnl > 0:
        tax = annual_realized_pnl * tax_rate
        cash -= tax
        results[-1]['cash'] = cash
    
    # 最終日時点での保有株式を時価評価
    final_date = prices_pivot.index.max()
    final_portfolio_value = 0
    
    if len(portfolio) > 0:
        for code, position in portfolio.items():
            if code in prices_pivot.columns:
                final_price = prices_pivot.loc[final_date, code]
                if pd.notna(final_price):
                    final_portfolio_value += position['shares'] * final_price
        
        results[-1]['invested'] = final_portfolio_value
    
    print(f"\n戦略: {strategy_name} 完了")
    print(f"最終現金: {cash:,.0f}円")
    print(f"最終保有株式時価: {final_portfolio_value:,.0f}円")
    print(f"最終総資産: {(cash + final_portfolio_value):,.0f}円")
    
    return results

print("バックテスト関数定義完了")

## 9. 全戦略のバックテスト実行

In [ ]:
# 全戦略のバックテスト
strategies = {
    'baseline': 'ベースライン（ランダム）',
    'low_pbr': '戦略A: 低PBR',
    'high_roe': '戦略B: 高予想ROE',
    'positive_return': '戦略C: 過去6ヵ月リターン > 0',
    'composite': '戦略D: 複合戦略'
}

all_results = {}

for strategy_key, strategy_label in strategies.items():
    print(f"\n\n{'#'*60}")
    print(f"実行: {strategy_label}")
    print(f"{'#'*60}")
    
    results = backtest_strategy(
        strategy_name=strategy_key,
        rebalance_dates=rebalance_dates,
        prices_pivot=df_price_pivot,
        fin_data=fin_by_date,
        returns_6m_data=returns_6m_by_date
    )
    
    all_results[strategy_key] = results

print("\n\n全戦略のバックテスト完了")

## 10. パフォーマンス分析

In [ ]:
def calculate_metrics(results, initial_cash=10_000_000):
    """
    パフォーマンス指標を計算
    """
    df = pd.DataFrame(results)
    df['total_value'] = df['cash'] + df['invested']
    df['return'] = df['total_value'].pct_change()
    df['cumulative_return'] = (1 + df['return']).cumprod() - 1
    
    # 基本指標
    final_value = df['total_value'].iloc[-1]
    total_return = df['cumulative_return'].iloc[-1]
    
    # 年率換算
    years = (df['date'].iloc[-1] - df['date'].iloc[0]).days / 365.25
    annual_return = (1 + total_return) ** (1 / years) - 1
    annual_vol = df['return'].std() * np.sqrt(12)  # 月次リバランス
    
    # MDD
    df['peak'] = df['total_value'].cummax()
    df['drawdown'] = (df['total_value'] - df['peak']) / df['peak']
    mdd = df['drawdown'].min()
    
    # Sharpe, Calmar
    sharpe = df['return'].mean() / df['return'].std() * np.sqrt(12) if df['return'].std() > 0 else 0
    calmar = annual_return / abs(mdd) if mdd != 0 else 0
    
    return {
        'final_value': final_value,
        'total_return': total_return,
        'annual_return': annual_return,
        'annual_vol': annual_vol,
        'mdd': mdd,
        'sharpe': sharpe,
        'calmar': calmar,
        'years': years
    }

# 全戦略のメトリクス計算
metrics_summary = []

for strategy_key, strategy_label in strategies.items():
    metrics = calculate_metrics(all_results[strategy_key])
    metrics_summary.append({
        '戦略': strategy_label,
        '最終資産（円）': f"{metrics['final_value']:,.0f}",
        '総リターン（%）': f"{metrics['total_return']*100:.2f}",
        '年率リターン（%）': f"{metrics['annual_return']*100:.2f}",
        '年率ボラティリティ（%）': f"{metrics['annual_vol']*100:.2f}",
        'MDD（%）': f"{metrics['mdd']*100:.2f}",
        'シャープレシオ': f"{metrics['sharpe']:.2f}",
        'カルマー比': f"{metrics['calmar']:.2f}"
    })

df_metrics = pd.DataFrame(metrics_summary)

print("\n" + "="*80)
print("全戦略パフォーマンス比較")
print("="*80)
display(df_metrics)

## 11. 増配実績の検証（精度評価）

In [ ]:
print("増配実績の検証を開始...")

# 配当データの履歴を取得
div_history = df_fin_clean[['disclosed_date', 'code', 'DivAnn']].copy() if 'DivAnn' in df_fin_clean.columns else pd.DataFrame()

if len(div_history) == 0:
    print("警告: 配当データ（DivAnn）が存在しないため、増配実績の検証をスキップします")
else:
    # 各銘柄・年の配当データを整理
    div_history['year'] = div_history['disclosed_date'].dt.year
    div_annual = div_history.sort_values('disclosed_date').groupby(['code', 'year'])['DivAnn'].last().reset_index()
    
    # 前年との比較
    div_annual = div_annual.sort_values(['code', 'year'])
    div_annual['prev_div'] = div_annual.groupby('code')['DivAnn'].shift(1)
    div_annual['div_growth'] = (div_annual['DivAnn'] - div_annual['prev_div']) / div_annual['prev_div']
    div_annual['is_increased'] = div_annual['div_growth'] > 0
    
    print(f"配当履歴データ: {len(div_annual):,} 行")
    print(f"増配実績あり: {div_annual['is_increased'].sum():,} 件")
    
    # 各戦略の増配予測精度を計算
    accuracy_summary = []
    
    for strategy_key, strategy_label in strategies.items():
        results = all_results[strategy_key]
        
        total_stocks = 0
        total_increased = 0
        
        for i, row in enumerate(results):
            if len(row['portfolio']) == 0:
                continue
            
            rebalance_date = row['date']
            current_year = rebalance_date.year
            next_year = current_year + 1
            
            # 選定した銘柄
            selected_codes = list(row['portfolio'].keys())
            
            # 翌年の増配実績を確認
            for code in selected_codes:
                # 当年の配当
                current_div = div_annual[(div_annual['code'] == code) & (div_annual['year'] == current_year)]
                # 翌年の配当
                next_div = div_annual[(div_annual['code'] == code) & (div_annual['year'] == next_year)]
                
                if len(current_div) > 0 and len(next_div) > 0:
                    total_stocks += 1
                    if next_div['is_increased'].iloc[0]:
                        total_increased += 1
        
        accuracy = (total_increased / total_stocks * 100) if total_stocks > 0 else 0
        
        accuracy_summary.append({
            '戦略': strategy_label,
            '選定銘柄数（延べ）': total_stocks,
            '翌年増配した銘柄数': total_increased,
            '増配予測的中率（%）': f"{accuracy:.2f}"
        })
    
    df_accuracy = pd.DataFrame(accuracy_summary)
    
    print("\n" + "="*80)
    print("増配予測精度の評価")
    print("="*80)
    display(df_accuracy)

## 12. 結果の保存

In [ ]:
# 結果をCSVに保存
output_dir = PROJECT_ROOT / 'analyses/20260221_0930_quants_model_dividend_growth'

# バックテスト結果（全戦略の日次データ）
all_daily_results = []
for strategy_key, strategy_label in strategies.items():
    df_strategy = pd.DataFrame(all_results[strategy_key])
    df_strategy['total_value'] = df_strategy['cash'] + df_strategy['invested']
    df_strategy['strategy'] = strategy_label
    all_daily_results.append(df_strategy[['date', 'strategy', 'total_value', 'n_stocks', 'cash', 'invested']])

df_all_daily = pd.concat(all_daily_results, ignore_index=True)
df_all_daily.to_csv(output_dir / 'backtest_results.csv', index=False, encoding='utf-8-sig')
print(f"結果を保存: {output_dir / 'backtest_results.csv'}")

# 評価指標をJSON形式で保存
metrics_dict = {}
for strategy_key, strategy_label in strategies.items():
    metrics = calculate_metrics(all_results[strategy_key])
    metrics_dict[strategy_label] = {
        'final_value': float(metrics['final_value']),
        'total_return': float(metrics['total_return']),
        'annual_return': float(metrics['annual_return']),
        'annual_volatility': float(metrics['annual_vol']),
        'max_drawdown': float(metrics['mdd']),
        'sharpe_ratio': float(metrics['sharpe']),
        'calmar_ratio': float(metrics['calmar'])
    }

with open(output_dir / 'backtest_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, indent=2, ensure_ascii=False)
print(f"評価指標を保存: {output_dir / 'backtest_metrics.json'}")

# パフォーマンスサマリをテキストで保存
with open(output_dir / 'performance_summary.txt', 'w', encoding='utf-8') as f:
    f.write("増配候補銘柄抽出モデル バックテスト結果\n")
    f.write("="*80 + "\n\n")
    f.write("全戦略パフォーマンス比較\n")
    f.write("-"*80 + "\n")
    f.write(df_metrics.to_string(index=False))
    f.write("\n\n")
    
    if len(div_history) > 0:
        f.write("増配予測精度の評価\n")
        f.write("-"*80 + "\n")
        f.write(df_accuracy.to_string(index=False))
        f.write("\n")

print(f"サマリを保存: {output_dir / 'performance_summary.txt'}")

# 増配予測精度をCSVで保存
if len(div_history) > 0:
    df_accuracy.to_csv(output_dir / 'dividend_prediction_accuracy.csv', index=False, encoding='utf-8-sig')
    print(f"増配予測精度を保存: {output_dir / 'dividend_prediction_accuracy.csv'}")

print("\n完了！")